# Part 2 - New Data
- At this point, I have completed the initial analysis for the project
- I found that educational related features seem to be the most important features to predicting the target
- Now...what about other features, like crime or political ideology?
- Basically, I want to try everything I just did, but removing the less important features and adding the new ones

## Data Gathering + Cleaning: Crime
- For the following section, I will be getting all the data from [Crime/Law Enforcement Stats (Uniform Crime Reporting Program)](https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr)

In [1]:
import pandas as pd

### 2010
- The data comes from this link: [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2010/crime-in-the-u.s.-2010/tables/10tbl05.xls)
- There is a note on the state Minnesota, saying: *"The data collection methodology for the offense of forcible rape used by the Minnesota state Uniform Crime Reporting (UCR) Program (with the exception of Minneapolis and St. Paul, Minnesota) does not comply with national UCR Program guidelines. Consequently, its figures for forcible rape and violent crime (of which forcible rape is a part) are not published in this table."*

In [7]:
crime2010 = pd.read_excel(
    "data/crime/2010.xls",
    engine="xlrd",
    sheet_name=0,
    skiprows=3
)

# cleaning column names + dropping empty columns
crime2010.columns = crime2010.columns.str.replace('\n', ' ', regex=False)
crime2010 = crime2010.dropna(axis=1, how='all')

# forward filling the state and area data (it is missing otherwise)
crime2010['State'] = crime2010['State'].ffill()
crime2010['Area'] = crime2010['Area'].ffill()

# only keeping the state total rows
crime2010 = crime2010[
    (crime2010['Area'] == 'State Total') &
    (crime2010['Unnamed: 2'].isna() | (crime2010['Unnamed: 2'].str.strip() == ''))
].copy()

# removing footnote
crime2010 = crime2010[crime2010['State'].str.len() < 20]

# idk what these columns do...hopefully they are not important
crime2010 = crime2010.drop(columns=['Area', 'Unnamed: 2', 'Unnamed: 13'])

# getting rid of footote numbers
crime2010['State'] = crime2010['State'].str.replace(r'\d+$', '', regex=True).str.strip()

crime2010 = crime2010.reset_index(drop=True)

In [8]:
crime2010.head()

,State,Population,Violent crime,Murder and nonnegligent manslaughter,Forcible rape,Robbery,Aggravated assault,Property crime,Burglary,Larceny- theft,Motor vehicle theft
0,ALABAMA,4779736.0,18056,273,1349,4761,11673,168092,42034,115458.0,10600.0
1,ALASKA,710231.0,4537,31,533,594,3379,20259,3105,15535.0,1619.0
2,ARIZONA,6392017.0,26085,409,2165,6937,16574,225893,50771,153614.0,21508.0
3,ARKANSAS,2915918.0,14735,138,1312,2372,10913,103775,32511,65720.0,5544.0
4,CALIFORNIA,37253956.0,164133,1809,8331,58116,95877,981939,228857,600558.0,152524.0


In [21]:
# making a function to make everything easier

def cleaning_crime(filepath, space=' '):
    df = pd.read_excel(
        filepath,
        engine="xlrd",
        sheet_name=0,
        skiprows=3
    )
    # cleaning column names + dropping empty columns
    df.columns = df.columns.str.replace('\n', space, regex=False)
    df = df.dropna(axis=1, how='all')
    # forward filling the state and area data (it is missing otherwise)
    df['State'] = df['State'].ffill()
    df['Area'] = df['Area'].ffill()
    # only keeping the state total rows
    df = df[
        (df['Area'] == 'State Total') &
        (df['Unnamed: 2'].isna() | (df['Unnamed: 2'].str.strip() == ''))
    ].copy()
    # removing footnote
    df = df[df['State'].str.len() < 20]
    # idk what these columns do...hopefully they are not important
    df = df.drop(columns=['Area', 'Unnamed: 2'])
    # getting rid of footote numbers
    df['State'] = df['State'].str.replace(r'\d+$', '', regex=True).str.strip()
    df = df.reset_index(drop=True)
    
    return df

### 2011
- The data from this section comes from [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2011/crime-in-the-u.s.-2011/tables/table-5)
- There is a footnote on **Alabama** and **Montana**, saying, *"Because of changes in the state's reporting practices, figures are not comparable to previous years' data."*
- There is also a footnote on **Minnesota**, saying *"The data collection methodology for the offense of forcible rape used by the Minnesota state Uniform Crime Reporting (UCR) Program (with the exception of Minneapolis and St. Paul, Minnesota) does not comply with national UCR Program guidelines. Consequently, its figures for forcible rape and violent crime (of which forcible rape is a part) are not published in this table."*

In [30]:
filepath2011 = "data/crime/2011.xls"

crime2011 = cleaning_crime(filepath2011, space=' ')

In [31]:
crime2011.head()

,State,Population,Violent crime,Murder and nonnegligent manslaughter,Forcible rape,Robbery,Aggravated assault,Property crime,Burglary,Larceny- theft,Motor vehicle theft
0,ALABAMA,4802740.0,20174,301,1371,4910,13592,173190,51117,111411.0,10662.0
1,ALASKA,722718.0,4383,29,420,576,3358,19028,2826,14859.0,1343.0
2,ARIZONA,6482505.0,26311,405,2264,7127,16515,230422,54929,155664.0,19829.0
3,ARKANSAS,2937979.0,14129,162,1213,2428,10326,110295,34471,70012.0,5812.0
4,CALIFORNIA,37691912.0,154944,1792,7663,54292,91197,973901,230090,596963.0,146848.0


### 2012
- The data from this section was downloaded from this link: [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2012/crime-in-the-u.s.-2012/tables/5tabledatadecpdf/table_5_crime_in_the_united_states_by_state_2012.xls)
- There is a footnote on **Minnesota**, saying, *"The data collection methodology for the offense of forcible rape used by the Minnesota state Uniform Crime Reporting (UCR) Program (with the exception of Minneapolis and St. Paul, Minnesota) does not comply with national UCR Program guidelines. Consequently, its figures for forcible rape and violent crime (of which forcible rape is a part) are not published in this table."*
- There is also a footnote on **New York**, saying, *"Because of changes in the state/local agency's reporting practices, figures are not comparable to previous years' data."*

In [28]:
filepath2012 = "data/crime/2012.xls"

crime2012 = cleaning_crime(filepath2012)

In [29]:
crime2012.head()

,State,Population,Violent crime,Murder and nonnegligent manslaughter,Forcible rape,Robbery,Aggravated assault,Property crime,Burglary,Larceny- theft,Motor vehicle theft
0,ALABAMA,4822023.0,21693,342,1296,5020,15035,168878,47481,111523.0,9874.0
1,ALASKA,731449.0,4412,30,583,630,3169,20037,2950,15565.0,1522.0
2,ARIZONA,6553255.0,28108,358,2277,7386,18087,231930,52934,159838.0,19158.0
3,ARKANSAS,2949131.0,13835,173,1247,2320,10095,107941,31890,70327.0,5724.0
4,CALIFORNIA,38041430.0,160944,1884,7837,56521,94702,1049465,245767,635090.0,168608.0


### 2013
- The data from this section was downloaded from this link: [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2013/crime-in-the-u.s.-2013/tables/5tabledatadecpdf/table_5_crime_in_the_united_states_by_state_2013.xls)
- The **Violent Crime** section has a footnote, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the new **Rape (revised definition)** column, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the new **Rape (legacy definition)**, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- Alabama, Alaska, Arkansas, Colorado, Delaware, Florida, Idaho, Iowa, Michigan, Minnesota, Missouri, Montana, New Hampshire, North Dakota, Pennsylvania, Rhode Island, South Carolina, South Dakota, Tennessee, Vermont, Virginia, and West Virginia all have a footnote, saying, *"This state's agencies submitted rape data according to the revised UCR definition of rape."*
- Arizona, Connecticut, Illinois, Indiana, Kansas, Kentucky, Louisiana, Maine, Massachusetts, Mississippi, Nebraska, Ohio, Oklahoma, Oregon, Texas, Utah, Washington, and Wisconsin all have a footnote, saying, *"Agencies within this state submitted rape data according to both the revised UCR definition of rape and the legacy UCR definition of rape."*
- Arizona, Maine, Pennsylvania, Washington, and Wisconsin all have footnotes, saying, *"Because of changes in the state/local agency's reporting practices, figures are not comparable to previous years' data."*

yeesh...this is not looking too good

In [32]:
filepath2013 = "data/crime/2013.xls"

crime2013 = cleaning_crime(filepath2013)

In [33]:
crime2013.head()

,State,Population,Violent crime1,Murder and nonnegligent manslaughter,Rape (revised definition)2,Rape (legacy definition)3,Robbery,Aggravated assault,Property crime,Burglary,Larceny- theft,Motor vehicle theft
0,ALABAMA,4833722.0,20826,347,2044.0,1428.0,4648,13787,161993,42429,108993.0,10571.0
1,ALASKA,735132.0,4708,34,922.0,644.0,624,3128,21210,2916,16599.0,1695.0
2,"ARIZONA5,",6626624.0,27599,357,3050.0,2343.0,6702,17490,225243,48533,159272.0,17438.0
3,ARKANSAS,2959373.0,13621,159,1423.0,993.0,2258,9781,106613,30485,70450.0,5678.0
4,CALIFORNIA,38332521.0,154129,1746,9714.0,7464.0,53640,89029,1018907,232058,621557.0,165292.0


### 2014
- The data from this table was found from [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2014/crime-in-the-u.s.-2014/tables/table-5)
- The **Violent Crime** section has a footnote, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (revised definition)** column, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (legacy definition)**, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- Alabama, Alaska, Arizona, Arkansas, Colorado, Connecticut, Delaware, Florida, Hawaii, Idaho, Illinois, Iowa, Kansas, Kentucky, Louisiana, Michigan, Minnesota, Missouri, Montana, Nevada, New Hampshire, North Dakota, Oklahoma, Pennsylvania, South Carolina, South Dakota, Tennessee, Texas, Vermont, Virginia, West Virginia, and Wyoming all have a footnote, saying, *"This state's agencies submitted rape data according to the revised UCR definition of rape."*
- California, Indiana, Maine, Massachusetts, Mississippi, Nebraska, New York, Ohio, Oregon, Rhode Island, Utah, Washington, and Wisconsin all have a footnote, saying, *Agencies within this state submitted rape data according to both the revised UCR definition of rape and the legacy UCR definition of rape.*
- Georgia has a footnote, saying, *Because of changes in the state/local agency's reporting practices, figures are not comparable to previous years' data.*

In [36]:
filepath2014 = "data/crime/2014.xls"

crime2014 = cleaning_crime(filepath2014, space='')

In [37]:
crime2014.head()

,State,Population,Violent crime1,Murder and nonnegligent manslaughter,Rape(reviseddefinition)2,Rape(legacydefinition)3,Robbery,Aggravated assault,Property crime,Burglary,Larceny-theft,Motor vehicle theft
0,ALABAMA,4849377.0,20727,276,2005.0,1436.0,4701,13745,154094,39715,104238.0,10141.0
1,ALASKA,736732.0,4684,41,771.0,555.0,629,3243,20334,3150,15445.0,1739.0
2,ARIZONA,6731484.0,26916,319,3378.0,2464.0,6249,16970,215240,43562,154091.0,17587.0
3,ARKANSAS,2966369.0,14243,165,1763.0,1182.0,2050,10265,99018,24790,68627.0,5601.0
4,CALIFORNIA,38802500.0,153709,1699,11527.0,8398.0,48680,91803,947192,202670,592670.0,151852.0


### 2015
- The data from this table was located through this link: [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2015/crime-in-the-u.s.-2015/tables/table-5)
- The **Violent Crime** section has a footnote, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (revised definition)** column, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (legacy definition)**, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- Georgia, New Mexico, and North Carolina all have a footnote, saying, *"This state's agencies submitted rape data according to the legacy UCR definition of rape."*
- Indiana, Maine, Mississippi, Nebraska, New York, Ohio, Oregon, Rhode Island, Utah, Washington, and Wisconsin all have a footnote, saying, *"Agencies within this state submitted rape data according to both the revised UCR definition of rape and the legacy UCR definition of rape."*

In [39]:
filepath2015 = "data/crime/2015.xls"

crime2015 = cleaning_crime(filepath2015, space='')

In [41]:
crime2015.head()

,State,Population,Violentcrime1,Murder and nonnegligent manslaughter,Rape(reviseddefinition)2,Rape(legacydefinition)3,Robbery,Aggravated assault,Property crime,Burglary,Larceny-theft,Motor vehicle theft
0,ALABAMA,4858979.0,22952,348,2039.0,1456.0,4611,15954,144746,35255,99156.0,10335.0
1,ALASKA,738432.0,5392,59,901.0,650.0,761,3671,20806,3511,15249.0,2046.0
2,ARIZONA,6828065.0,28012,309,3108.0,2294.0,6360,18235,207107,37957,152365.0,16785.0
3,ARKANSAS,2978204.0,15526,181,1931.0,1300.0,2098,11316,96836,22640,68424.0,5772.0
4,CALIFORNIA,39144818.0,166883,1861,12811.0,9387.0,52862,99349,1024914,197404,656517.0,170993.0


### 2016
- The following data was found from this link: [Crime in the United States](https://ucr.fbi.gov/crime-in-the-u.s/2016/crime-in-the-u.s.-2016/topic-pages/tables/table-3)
- The **Violent Crime** section has a footnote, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (revised definition)** column, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- There is a footnote on the **Rape (legacy definition)**, saying, *"The violent crime figures include the offenses of murder, rape (revised definition), robbery, and aggravated assault."*
- Georgia has a footnote, saying, *"This state's agencies submitted rape data according to the legacy UCR definition of rape."*
- Indiana, Mississippi, Nebraska, New Mexico, New York, North Carolina, Ohio, Oklahoma, Oregon, Rhode Island, South Dakota, Washington, and Wisconsin all have a footnote, saying, *"Agencies within this state submitted rape data according to both the revised UCR definition of rape and the legacy UCR definition of rape."*

In [42]:
filepath2016 = "data/crime/2016.xls"

crime2016 = cleaning_crime(filepath2016, space='')

In [43]:
crime2016.head()

,State,Population,Violentcrime1,Murder and nonnegligent manslaughter,Rape(reviseddefinition)2,Rape(legacydefinition)3,Robbery,Aggravated assault,Property crime,Burglary,Larceny-theft,Motor vehicle theft,Unnamed: 15
0,ALABAMA,4863300.0,25886,407,1916.0,1385.0,4686.0,18877,143362,34065,97574.0,11723.0,NaN
1,ALASKA,741894.0,5966,52,1053.0,757.0,850.0,4011,24876,4053,17766.0,3057.0,NaN
2,ARIZONA,6931071.0,32583,380,3290.0,2452.0,7055.0,21858,206432,37736,150275.0,18421.0,NaN
3,ARKANSAS,2988248.0,16461,216,2143.0,1545.0,2120.0,11982,97673,23771,66747.0,7155.0,NaN
4,CALIFORNIA,39250017.0,174796,1930,13702.0,10149.0,54789.0,104375,1002070,188304,637010.0,176756.0,NaN
